In [1]:
import numpy as np, pandas as pd
from scipy.stats import binomtest
R = "results/rebuttal_main_benchmark"
ot   = pd.read_csv(f"{R}/otica.csv")
base = pd.read_csv(f"{R}/baselines.csv")
fr   = pd.read_csv(f"{R}/fastica_runs.csv")
KEY = ["Dim", "Config", "Trial"]

# FastICA as in the paper: best of 50 runs by (oracle) Amari error to the true A
fast_oracle = fr.loc[fr.groupby(KEY)["Amari"].idxmin()].assign(Method="FastICA")
# FastICA with an unsupervised pick: the run with the largest negentropy proxy (its own objective)
fast_unsup = fr.loc[fr.groupby(KEY)["J"].idxmax()].assign(Method="FastICA (own-objective pick)")
# FastICA single run (random_state = trial*1000)
fast_single = fr[fr.Run == 0].assign(Method="FastICA (1 run)")
# FastICA at matched wall-clock: runs in order until their total time exceeds OT-ICA's time, pick by own objective
fr_t = fr.merge(ot[KEY + ["Time"]].rename(columns={"Time": "T_ot"}), on=KEY)
fr_t["cum"] = fr_t.sort_values("Run").groupby(KEY)["Time"].cumsum()
within = fr_t[(fr_t.cum <= fr_t.T_ot) | (fr_t.Run == 0)]
fast_matched = within.loc[within.groupby(KEY)["J"].idxmax()].assign(Method="FastICA (matched time)")
runs_matched = within.groupby(KEY).size().rename("R_matched").reset_index()

df = pd.concat([ot[KEY + ["Method", "Amari", "Time"]], base[KEY + ["Method", "Amari", "Time"]],
                fast_oracle[KEY + ["Method", "Amari"]], fast_unsup[KEY + ["Method", "Amari"]],
                fast_single[KEY + ["Method", "Amari"]], fast_matched[KEY + ["Method", "Amari"]]], ignore_index=True)
df["Amari_norm"] = df["Amari"] / (df["Dim"] - 1)          # Amari index normalized to [0, 1]
fast_time = fr.groupby(KEY)["Time"].sum().rename("Time").reset_index().assign(Method="FastICA")
print("trials per method:", df.groupby("Method").size().to_dict())


trials per method: {'FastICA': 150, 'FastICA (1 run)': 150, 'FastICA (matched time)': 150, 'FastICA (own-objective pick)': 150, 'InfoMax': 150, 'JADE': 150, 'OT-ICA': 150, 'Picard': 150}


In [2]:
METHODS = ["OT-ICA", "FastICA", "JADE", "InfoMax", "Picard"]
tab = df[df.Method.isin(METHODS)].pivot_table(index=["Config", "Dim"], columns="Method", values="Amari_norm", aggfunc="mean")[METHODS]
print("\nMean NORMALIZED Amari (unclipped):\n", tab.round(4).to_string())
raw = df[df.Method.isin(METHODS)].pivot_table(index=["Config", "Dim"], columns="Method", values="Amari", aggfunc="mean")[METHODS]
print("\nMean RAW Amari (paper scale, unclipped):\n", raw.round(3).to_string())



Mean NORMALIZED Amari (unclipped):
 Method                       OT-ICA  FastICA    JADE  InfoMax  Picard
Config                  Dim                                          
Continuous Only         10   0.0056   0.0102  0.0162   0.0111  0.0104
                        20   0.0056   0.0096  0.0169   0.0099  0.0097
                        30   0.0055   0.0094  0.0169   0.0096  0.0094
Discrete Only           10   0.0638   0.0681  0.0492   0.0893  0.0966
                        20   0.0822   0.0996  0.0773   0.1303  0.1247
                        30   0.1173   0.1162  0.1190   0.1449  0.1366
Full Hybrid             10   0.0051   0.0284  0.0240   0.0330  0.0308
                        20   0.0120   0.0233  0.0301   0.0314  0.0319
                        30   0.0128   0.0228  0.0325   0.0270  0.0316
Strictly Super-Gaussian 10   0.0064   0.0099  0.0162   0.0114  0.0100
                        20   0.0060   0.0096  0.0166   0.0100  0.0097
                        30   0.0061   0.0096  0.0166 

In [3]:
def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

wide = df.pivot_table(index=KEY, columns="Method", values="Amari_norm")
rng = np.random.default_rng(0)
rows = []
for b in ["FastICA", "JADE", "InfoMax", "Picard"]:
    for (cfg, d), g in wide.groupby(["Config", "Dim"]):
        diff = (g[b] - g["OT-ICA"]).dropna()          # > 0 means OT-ICA is better
        nz = diff[diff != 0]
        wins = int((nz > 0).sum()); n = len(nz)
        p = binomtest(wins, n, 0.5).pvalue if n else np.nan
        boot = [np.median(rng.choice(diff.values, len(diff))) for _ in range(2000)]
        rows.append(dict(Baseline=b, Config=cfg, Dim=d, wins=wins, n=n, p=p, median_diff=np.median(diff),
                         ci_lo=np.percentile(boot, 2.5), ci_hi=np.percentile(boot, 97.5)))
st = pd.DataFrame(rows)
st["p_holm_within_baseline"] = st.groupby("Baseline")["p"].transform(holm)
st["p_holm_all"] = holm(st["p"])
pd.set_option("display.width", 200)
print(st.round(4).to_string(index=False))
print("\nCells significant after Holm (alpha=0.05), OT-ICA better:")
for col in ["p_holm_within_baseline", "p_holm_all"]:
    sig = st[(st[col] < 0.05) & (st.median_diff > 0)]
    print(f"  {col}: {len(sig)} / {len(st)}", sig.groupby("Baseline").size().to_dict())

# pooled aggregate test: all paired trials in the continuous/heterogeneous regimes
CONT = ["Continuous Only", "Strictly Super-Gaussian", "Zero Gaussian", "Full Hybrid"]
print("\nPooled sign test over all trials of the continuous/heterogeneous regimes:")
for b in ["FastICA", "JADE", "InfoMax", "Picard"]:
    w = wide[wide.index.get_level_values("Config").isin(CONT)]
    diff = (w[b] - w["OT-ICA"]).dropna(); nz = diff[diff != 0]
    print(f"  vs {b:8s}: OT-ICA better in {(nz > 0).sum()}/{len(nz)} trials, p = {binomtest(int((nz > 0).sum()), len(nz)).pvalue:.1e}")


Baseline                  Config  Dim  wins  n      p  median_diff   ci_lo  ci_hi  p_holm_within_baseline  p_holm_all
 FastICA         Continuous Only   10    10 10 0.0020       0.0044  0.0032 0.0059                  0.0293      0.1172
 FastICA         Continuous Only   20    10 10 0.0020       0.0045  0.0029 0.0049                  0.0293      0.1172
 FastICA         Continuous Only   30    10 10 0.0020       0.0043  0.0029 0.0044                  0.0293      0.1172
 FastICA           Discrete Only   10     5 10 1.0000       0.0049 -0.0234 0.0365                  1.0000      1.0000
 FastICA           Discrete Only   20     8 10 0.1094       0.0103  0.0012 0.0390                  0.3281      0.9844
 FastICA           Discrete Only   30     3 10 0.3438      -0.0068 -0.0095 0.0006                  0.6875      1.0000
 FastICA             Full Hybrid   10    10 10 0.0020       0.0180  0.0070 0.0397                  0.0293      0.1172
 FastICA             Full Hybrid   20     9 10 0.0215   

In [4]:
FV = ["OT-ICA", "FastICA", "FastICA (own-objective pick)", "FastICA (matched time)", "FastICA (1 run)"]
print(df[df.Method.isin(FV)].pivot_table(index=["Config", "Dim"], columns="Method", values="Amari_norm", aggfunc="mean")[FV].round(4).to_string())
print("\nFastICA runs that fit in OT-ICA's wall-clock (median per cell):")
print(runs_matched.groupby(["Config", "Dim"])["R_matched"].median().unstack().to_string())
tim = pd.concat([ot[KEY + ["Method", "Time"]], base[KEY + ["Method", "Time"]], fast_time])
print("\nMedian wall-clock per trial [s] (OT-ICA on GPU; baselines on 1 CPU core; FastICA = all 50 runs):")
print(tim.pivot_table(index="Dim", columns="Method", values="Time", aggfunc="median").round(2).to_string())
print("\nMedian time of ONE FastICA run [s]:", fr.groupby("Dim")["Time"].median().round(3).to_dict())
print("\nFraction of FastICA runs hitting max_iter=10000:")
print((fr.assign(cap=fr.n_iter >= 10000).groupby(["Config", "Dim"])["cap"].mean()).unstack().round(2).to_string())


Method                       OT-ICA  FastICA  FastICA (own-objective pick)  FastICA (matched time)  FastICA (1 run)
Config                  Dim                                                                                        
Continuous Only         10   0.0056   0.0102                        0.0104                  0.0104           0.0104
                        20   0.0056   0.0096                        0.0097                  0.0097           0.0097
                        30   0.0055   0.0094                        0.0094                  0.0094           0.0094
Discrete Only           10   0.0638   0.0681                        0.0811                  0.0797           0.0826
                        20   0.0822   0.0996                        0.1167                  0.1253           0.1253
                        30   0.1173   0.1162                        0.1306                  0.1332           0.1332
Full Hybrid             10   0.0051   0.0284                        0.03

In [5]:
ab = pd.read_csv("results/rebuttal_contrast_ablation/ablation.csv")
both = pd.concat([ot[KEY + ["Method", "Amari"]], ab[KEY + ["Method", "Amari"]]])
both = both[both.set_index(KEY).index.isin(ab.set_index(KEY).index)]   # only mixtures that E2 has finished
both["Amari_norm"] = both["Amari"] / (both["Dim"] - 1)
V = ["OT-ICA", "OT-ICA K=1", "Logcosh pipeline"]
print(both.pivot_table(index=["Config", "Dim"], columns="Method", values="Amari_norm", aggfunc="mean")[V].round(4).to_string())
wab = both.pivot_table(index=KEY, columns="Method", values="Amari_norm")
for v in ["Logcosh pipeline", "OT-ICA K=1"]:
    for name, mask in [("continuous/hybrid", wab.index.get_level_values("Config").isin(CONT)), ("all regimes", slice(None))]:
        diff = (wab[v] - wab["OT-ICA"])[mask].dropna(); nz = diff[diff != 0]
        if len(nz):
            print(f"OT-ICA better than {v:16s} [{name}]: {(nz > 0).sum()}/{len(nz)} trials, p = {binomtest(int((nz > 0).sum()), len(nz)).pvalue:.1e}")


Method                       OT-ICA  OT-ICA K=1  Logcosh pipeline
Config                  Dim                                      
Continuous Only         10   0.0056      0.0180            0.0113
                        20   0.0056      0.0354            0.0114
                        30   0.0055      0.0731            0.0125
Discrete Only           10   0.0638      0.2191            0.1347
                        20   0.0822      0.2893            0.1706
                        30   0.1173      0.3163            0.1735
Full Hybrid             10   0.0051      0.0533            0.0382
                        20   0.0120      0.1118            0.0367
                        30   0.0128      0.1365            0.0404
Strictly Super-Gaussian 10   0.0064      0.0105            0.0105
                        20   0.0060      0.0171            0.0105
                        30   0.0061      0.0227            0.0106
Zero Gaussian           10   0.0046      0.0299            0.0185
          

In [ ]:
rad = pd.read_csv("results/rebuttal_radical/radical.csv")
e3 = pd.concat([df[df.Method.isin(["OT-ICA", "FastICA"])],
                rad.assign(Amari_norm=rad.Amari / (rad.Dim - 1))[KEY + ["Method", "Amari", "Amari_norm"]]])
e3 = e3[e3.set_index(KEY).index.isin(rad.set_index(KEY).index)]   # only mixtures RADICAL has finished
print(e3.pivot_table(index=["Config", "Dim"], columns="Method", values="Amari_norm", aggfunc="mean")[["OT-ICA", "RADICAL", "FastICA"]].round(4).to_string())
print("\nmedian RADICAL time per trial [s]:", rad.groupby("Dim")["Time"].median().round(0).to_dict())
w3 = e3.pivot_table(index=KEY, columns="Method", values="Amari_norm")
rr = []
for (cfg, d), g in list(w3.groupby(["Config", "Dim"])) + [((c, "all d"), g) for c, g in w3.groupby("Config")]:
    diff = (g["RADICAL"] - g["OT-ICA"]).dropna(); nz = diff[diff != 0]; k = int((nz > 0).sum())
    rr.append(dict(Config=cfg, Dim=d, wins=k, n=len(nz), p=binomtest(k, len(nz)).pvalue if len(nz) else np.nan,
                   ratio=g["RADICAL"].mean() / g["OT-ICA"].mean()))
rr = pd.DataFrame(rr); pooled = rr.Dim == "all d"
rr.loc[pooled, "p_holm"] = holm(rr.loc[pooled, "p"])
print("\nOT-ICA vs RADICAL (wins = trials where OT-ICA is better; ratio = mean error RADICAL / OT-ICA):")
print(rr.round(4).to_string(index=False))
